task 1: Basic Data Exploration and Cleaning using Pandas

## Step 1: Load Data

Import required libraries and load the dataset.

In [ ]:
import os
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set(style='whitegrid')

def load_dataset(preferred_path=None):
    """Load dataset from Colab or local path."""
    try:
        from google.colab import files
        print('Uploading from Colab...')
        uploaded = files.upload()
        fname = next(iter(uploaded))
        return pd.read_csv(io.BytesIO(uploaded[fname]), low_memory=False)
    except Exception:
        if preferred_path and os.path.exists(preferred_path):
            return pd.read_csv(preferred_path, low_memory=False)
        alt = os.path.join('archive(2)', 'Combined_dataset.csv')
        if os.path.exists(alt):
            return pd.read_csv(alt, low_memory=False)
        raise FileNotFoundError('Dataset not found.')

df = load_dataset()
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Columns: {list(df.columns[:5])}... and {len(df.columns) - 5} more')

## Step 2: Understand Data

Examine data types, missing values, and summary statistics.

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\n' + '='*50)

print('\nMissing Values (%)')
missing_pct = (df.isna().sum() / len(df)) * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False).head(15)
for col, pct in missing_pct.items():
    print(f'{col}: {pct:.1f}%')

print('\nBasic Statistics:')
display(df.describe(include=[np.number]).T)

## Step 3: Data Cleaning

Handle missing values, convert data types, remove duplicates.

In [ ]:
def coerce_numeric(series):
    """Convert string/mixed columns to numeric."""
    s = series.astype(str).str.strip().replace({'': None, 'nan': None})
    s = s.str.replace(r'[^0-9\\.\\-]', '', regex=True)
    return pd.to_numeric(s, errors='coerce')

numeric_cols = ['initial_price', 'final_price', 'discount', 'rating', 'ratings_count']
for col in numeric_cols:
    if col in df.columns:
        df[col] = coerce_numeric(df[col])

if 'ratings_count' in df.columns:
    df['ratings_count'] = df['ratings_count'].fillna(0).astype(int)
if 'rating' in df.columns:
    df['rating'] = df['rating'].fillna(df['rating'].median())

before = len(df)
df.drop_duplicates(inplace=True)
if 'product_id' in df.columns:
    df.drop_duplicates(subset=['product_id'], inplace=True)
after = len(df)

print(f'Duplicates removed: {before - after}')
print(f'Final shape: {df.shape}')

## Step 4: Feature Engineering

Create new features: price difference, discount %, popularity score.

In [ ]:
for col in ['initial_price', 'final_price']:
    if col not in df.columns:
        df[col] = np.nan

df['price_difference'] = df['initial_price'] - df['final_price']
df['discount_pct'] = np.where(
    df['initial_price'] > 0,
    (df['price_difference'] / df['initial_price']) * 100,
    np.nan
)
df['popularity_score'] = df['rating'].fillna(0) * df['ratings_count'].fillna(0)

print('Features created: price_difference, discount_pct, popularity_score')
cols = ['initial_price', 'final_price', 'price_difference', 'discount_pct', 'popularity_score']
cols = [c for c in cols if c in df.columns]
display(df[cols].head())

## Step 5: Analysis

### Univariate Analysis

In [ ]:
analysis_vars = ['final_price', 'rating', 'ratings_count', 'discount_pct']
for col in analysis_vars:
    if col not in df.columns:
        continue
    data = df[col].dropna()
    if len(data) == 0:
        continue
    print(f'{col}:')
    print(f'  Mean: {data.mean():.2f}, Median: {data.median():.2f}, Std: {data.std():.2f}')
    print(f'  Range: {data.min():.2f} - {data.max():.2f}')
    
    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)
    iqr = q3 - q1
    outliers = data[(data < q1 - 1.5*iqr) | (data > q3 + 1.5*iqr)]
    print(f'  Outliers: {len(outliers)} ({100*len(outliers)/len(data):.1f}%)')
    print()

### Bivariate Analysis

In [ ]:
corr_cols = ['initial_price', 'final_price', 'rating', 'ratings_count', 'discount_pct', 'popularity_score']
avail = [c for c in corr_cols if c in df.columns]

if len(avail) > 1:
    print('Key Correlations:')
    corr = df[avail].corr().abs().unstack().sort_values(ascending=False)
    print(corr.drop_duplicates().head(10))

print('\nVariable Relationships:')
pairs = [('final_price', 'rating'), ('discount_pct', 'popularity_score'), ('rating', 'ratings_count')]
for x, y in pairs:
    if x in df.columns and y in df.columns:
        r = df[[x, y]].dropna().corr().iloc[0, 1]
        print(f'{x} vs {y}: r = {r:.3f}')

### Category-Level Analysis

In [ ]:
if 'category' in df.columns:
    cat_stats = df.groupby('category', as_index=False).agg({
        'final_price': 'mean',
        'rating': 'mean',
        'discount_pct': 'mean',
        'product_id': 'count' if 'product_id' in df.columns else 'size'
    }).sort_values('product_id', ascending=False)
    print(f'Total categories: {len(cat_stats)}')
    print('\nTop 10 Categories:')
    display(cat_stats.head(10))
else:
    print('No category column found')

## Step 6: Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribution of Key Variables', fontsize=14)

vars_plot = ['final_price', 'rating', 'ratings_count', 'discount_pct']
for i, col in enumerate(vars_plot):
    if col in df.columns:
        ax = axes[i//2, i%2]
        df[col].dropna().hist(ax=ax, bins=30, edgecolor='black', alpha=0.7)
        ax.set_title(col)
        ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

### Histograms - Distribution of Key Variables

In [ ]:
if 'category' in df.columns:
    top_cats = df['category'].value_counts().head(10)
    plt.figure(figsize=(12, 5))
    top_cats.plot(kind='barh', color='steelblue')
    plt.title('Top 10 Categories by Product Count')
    plt.xlabel('Count')
    plt.tight_layout()
    plt.show()

### Bar Charts - Category Insights

In [ ]:
if 'category' in df.columns and 'rating' in df.columns:
    top_cats = df['category'].value_counts().head(8).index
    df_cats = df[df['category'].isin(top_cats)].groupby('category').agg({
        'rating': 'mean',
        'final_price': 'mean',
        'discount_pct': 'mean'
    }).reset_index()
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    axes[0].bar(range(len(df_cats)), df_cats['rating'], color='coral', alpha=0.8, edgecolor='black')
    axes[0].set_xticks(range(len(df_cats)))
    axes[0].set_xticklabels(df_cats['category'], rotation=45, ha='right')
    axes[0].set_title('Average Rating by Category')
    axes[0].set_ylabel('Rating')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    axes[1].bar(range(len(df_cats)), df_cats['final_price'], color='lightgreen', alpha=0.8, edgecolor='black')
    axes[1].set_xticks(range(len(df_cats)))
    axes[1].set_xticklabels(df_cats['category'], rotation=45, ha='right')
    axes[1].set_title('Average Price by Category')
    axes[1].set_ylabel('Price (Rs.)')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    axes[2].bar(range(len(df_cats)), df_cats['discount_pct'], color='lightyellow', alpha=0.8, edgecolor='black')
    axes[2].set_xticks(range(len(df_cats)))
    axes[2].set_xticklabels(df_cats['category'], rotation=45, ha='right')
    axes[2].set_title('Average Discount % by Category')
    axes[2].set_ylabel('Discount %')
    axes[2].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Boxplots: Distribution & Outliers', fontsize=14)

vars_box = ['final_price', 'rating', 'ratings_count', 'discount_pct']
for i, col in enumerate(vars_box):
    if col in df.columns:
        ax = axes[i//2, i%2]
        data = df[col].dropna()
        ax.boxplot(data, vert=True)
        ax.set_title(col)
        ax.set_ylabel('Value')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Boxplots - Detecting Outliers & Spread

In [ ]:
if 'category' in df.columns and 'final_price' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    top_5_cats = df['category'].value_counts().head(5).index
    df_top = df[df['category'].isin(top_5_cats)]
    
    df_top.boxplot(column='final_price', by='category', ax=axes[0])
    axes[0].set_title('Price Distribution by Top Categories')
    axes[0].set_xlabel('Category')
    axes[0].set_ylabel('Final Price')
    
    if 'rating' in df.columns:
        df_top.boxplot(column='rating', by='category', ax=axes[1])
        axes[1].set_title('Rating Distribution by Top Categories')
        axes[1].set_xlabel('Category')
        axes[1].set_ylabel('Rating')
    
    plt.suptitle('')
    plt.tight_layout()
    plt.show()

## Step 7: Key Insights & Save Results

In [ ]:
print('='*60)
print('SUMMARY OF FINDINGS')
print('='*60)

if 'final_price' in df.columns:
    print(f'\n1. Average Product Price: Rs. {df["final_price"].mean():.0f}')

if 'rating' in df.columns:
    print(f'2. Average Product Rating: {df["rating"].mean():.2f}/5')

if 'discount_pct' in df.columns:
    avg_disc = df['discount_pct'].mean()
    print(f'3. Average Discount: {avg_disc:.1f}%')

if 'category' in df.columns:
    print(f'4. Total Categories: {df["category"].nunique()}')
    top_cat = df['category'].value_counts().index[0]
    print(f'5. Top Category: {top_cat}')

print(f'\nDataset Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print('='*60)

In [ ]:
output_file = 'cleaned_dataset.csv'
df.reset_index(drop=True, inplace=True)
df.to_csv(output_file, index=False)
print(f'✓ Cleaned dataset saved: {output_file}')
print(f'  Rows: {len(df)}, Columns: {len(df.columns)}')

try:
    from google.colab import files
    files.download(output_file)
except:
    print(f'  Path: {os.path.abspath(output_file)}')